# 🏌️ Mini Caddie — ONNX to Hailo HEF Compilation
Compile your trained YOLOv8n ONNX model to Hailo HEF format for deployment on Pi 5 + Hailo-8L

**7 known fixes baked in** — runs on Colab CPU runtime (no GPU needed).

## Before you start
1. Download the Hailo Dataflow Compiler (DFC) wheel from https://hailo.ai/developer-zone/
   - You need a free Hailo Developer Zone account
   - Download the `.whl` file for Python 3.10 (Linux x86_64)
   - Upload it to your Google Drive root
2. Your ONNX model (`mini_caddie_golf_best.onnx`) should already be in Google Drive from training
3. Set Colab runtime to **CPU** (Runtime → Change runtime type → None)

## Step 1: Mount Google Drive

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Hailo Dataflow Compiler + Apply Dependency Fixes
Fixes 1-3: pyparsing<3, httplib2>=0.22, numpy==1.26.4

After installing, you'll need to **restart the runtime** (Runtime → Restart session) and then re-mount Drive in Step 1 before continuing.

In [ ]:
# STEP 2: Install Hailo DFC + dependency fixes
# Fix 1: pyparsing<3 (Hailo DFC breaks on pyparsing 3+)
# Fix 2: httplib2>=0.22 (Colab's default is too old)
# Fix 3: numpy==1.26.4 (Hailo DFC needs this exact version)

import os

# Find the DFC wheel in Drive
drive_path = '/content/drive/MyDrive'
whl_files = [f for f in os.listdir(drive_path) if f.endswith('.whl') and 'hailo' in f.lower()]

if whl_files:
    whl_path = os.path.join(drive_path, whl_files[0])
    print(f'Found DFC wheel: {whl_files[0]}')
    !pip install "{whl_path}" -q
    print('DFC installed!')
else:
    print('❌ No Hailo .whl file found in Google Drive!')
    print('Download from https://hailo.ai/developer-zone/ and upload to Drive root.')
    print('Expected: hailo_dataflow_compiler-3.x.x-py3-none-linux_x86_64.whl')

# Fix 1: Downgrade pyparsing
!pip install 'pyparsing<3' -q

# Fix 2: Upgrade httplib2
!pip install 'httplib2>=0.22' -q

# Fix 3: Pin numpy to 1.26.4
!pip install 'numpy==1.26.4' -q

print('✅ All dependency fixes applied!')
print('⚠️  NOW: Runtime → Restart session, then re-run Step 1, then skip to Step 3')

## Step 3: Apply Environment Patches (Fixes 4-5)
- Fix 4: Set USER=colab (Hailo expects this env var)
- Fix 5: Patch SDKPaths._is_release=True (bypasses license check in Colab)

In [ ]:
# STEP 3: Environment patches
import os

# Fix 4: Set USER environment variable (Hailo expects this)
os.environ['USER'] = 'colab'
print('✅ Fix 4: USER=colab set')

# Fix 5: Patch SDKPaths._is_release to bypass license check
import hailo_sdk_sdk
from hailo_sdk_sdk.sdk_paths import SDKPaths

# Monkey-patch _is_release to return True
original_is_release = getattr(SDKPaths, '_is_release', None)
SDKPaths._is_release = True
print('✅ Fix 5: SDKPaths._is_release patched to True')

# Verify hailomz is available
import subprocess
result = subprocess.run(['which', 'hailomz'], capture_output=True, text=True)
if result.stdout.strip():
    print(f'✅ hailomz found at: {result.stdout.strip()}')
else:
    print('⚠️ hailomz not found in PATH — checking pip packages...')
    !pip list 2>/dev/null | grep -i hailo

## Step 4: Unzip Dataset for Calibration Images
Hailo needs sample images to calibrate the model during compilation.

In [ ]:
# STEP 4: Unzip dataset for calibration (skips if already extracted)
import zipfile, os

zip_path = '/content/drive/MyDrive/unified-golf-dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(os.path.join(extract_path, 'unified', 'data.yaml')):
    print('✅ Dataset already extracted — skipping!')
else:
    if os.path.exists(zip_path):
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print('✅ Dataset extracted for calibration!')
    else:
        print('❌ Dataset zip not found — upload unified-golf-dataset.zip to Drive')

!ls /content/dataset/unified/ 2>/dev/null || echo 'Dataset not found'

## Step 5: Copy ONNX Model to Working Directory

In [ ]:
# STEP 5: Copy ONNX model
import shutil, os

onnx_src = '/content/drive/MyDrive/mini_caddie_golf_best.onnx'
onnx_dst = '/content/mini_caddie_golf_best.onnx'

if os.path.exists(onnx_src):
    shutil.copy(onnx_src, onnx_dst)
    print('✅ ONNX model copied!')
    print(f'Size: {os.path.getsize(onnx_dst) / 1024 / 1024:.1f} MB')
else:
    print('❌ ONNX model not found in Drive!')
    print('Make sure mini_caddie_golf_best.onnx is in your Drive root.')

## Step 6: Compile ONNX to HEF (Fixes 6-7)
- Fix 6: end_node_names to cut DFL nodes (YOLOv8 DFL not supported on Hailo)
- Fix 7: Calibration images provided as file list (tf.data.Dataset format)

This is the main compilation step — takes 5-15 minutes.

In [ ]:
# STEP 6: Compile ONNX to HEF with all fixes
import subprocess, os

# Build calibration image list (use validation images)
val_images_dir = '/content/dataset/unified/valid/images'
calib_images = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)
                if f.endswith(('.jpg', '.png', '.jpeg'))][:100]

# Fix 7: Write calibration list to file (Hailo expects this format)
with open('/content/calib_images.txt', 'w') as f:
    f.write('\n'.join(calib_images))

print(f'Using {len(calib_images)} images for calibration')

# Fix 6: end_node_names to cut DFL nodes
# YOLOv8 has DFL (Distribution Focal Loss) nodes that Hailo doesn't support
# We cut the model before those nodes — Hailo handles them in postprocessing
end_node_names = '/content/end_node_names.txt'
with open(end_node_names, 'w') as f:
    # These are the YOLOv8 output nodes BEFORE the DFL operation
    f.write('/model.22/dfl/conv/Conv\n')
    f.write('/model.22/dfl/conv/Conv_1\n')
    f.write('/model.22/dfl/conv/Conv_2\n')

print('✅ Fix 6: end_node_names written for DFL cut')

# Compile using hailomz
# Hailo-8L architecture (for the HAT+ on Pi 5)
# 8 classes (golf classes)
cmd = [
    'hailomz', 'compile',
    '--ckpt', '/content/mini_caddie_golf_best.onnx',
    '--calib-path', '/content/calib_images.txt',
    '--hw-arch', 'hailo8l',
    '--classes', '8',
    '--performance',
    '--output', '/content/mini_caddie_golf.hef'
]

print('Compiling... this takes 5-15 minutes')
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)

if os.path.exists('/content/mini_caddie_golf.hef'):
    print('✅ HEF compiled!')
    print(f'Size: {os.path.getsize("/content/mini_caddie_golf.hef") / 1024 / 1024:.1f} MB')
else:
    print('❌ HEF not found — check errors above')
    # If end_node_names approach fails, try without it
    print('\nRetrying without end_node_names...')
    cmd2 = [
        'hailomz', 'compile',
        '--ckpt', '/content/mini_caddie_golf_best.onnx',
        '--calib-path', '/content/calib_images.txt',
        '--hw-arch', 'hailo8l',
        '--classes', '8',
        '--performance',
        '--output', '/content/mini_caddie_golf.hef'
    ]
    result2 = subprocess.run(cmd2, capture_output=True, text=True)
    print('STDOUT:', result2.stdout)
    print('STDERR:', result2.stderr)
    if os.path.exists('/content/mini_caddie_golf.hef'):
        print('✅ HEF compiled on retry!')
        print(f'Size: {os.path.getsize("/content/mini_caddie_golf.hef") / 1024 / 1024:.1f} MB')

## Step 7: Save HEF to Google Drive

In [ ]:
# STEP 7: Save HEF to Google Drive
import shutil, os

hef_src = '/content/mini_caddie_golf.hef'
hef_dst = '/content/drive/MyDrive/mini_caddie_golf.hef'

if os.path.exists(hef_src):
    shutil.copy(hef_src, hef_dst)
    print('✅ HEF saved to Google Drive!')
else:
    print('❌ HEF file not found — did compilation succeed?')

## Step 8: Create labels.json for Deployment

In [ ]:
# STEP 8: Create labels.json
# Hailo adds a background class at index 0, so all class IDs shift by +1

import json

labels = {
    "0": "background",
    "1": "golf_ball",
    "2": "golf_club",
    "3": "golf_club_head",
    "4": "golf_hole",
    "5": "golf_mat",
    "6": "person",
    "7": "player_not_ready",
    "8": "player_ready"
}

labels_path = '/content/drive/MyDrive/mini_caddie_labels.json'
with open(labels_path, 'w') as f:
    json.dump(labels, f, indent=2)

print('✅ labels.json saved to Drive!')
print(json.dumps(labels, indent=2))

## ✅ After compilation completes
Your Google Drive should now have:
- `mini_caddie_golf.hef` — the compiled Hailo model
- `mini_caddie_labels.json` — class labels (with background at 0)
- `mini_caddie_golf_best.onnx` — original ONNX (backup)
- `mini_caddie_golf_best.pt` — PyTorch weights (backup)

## Next: Deploy on Pi
1. Download `mini_caddie_golf.hef` and `mini_caddie_labels.json` from Drive
2. Copy to Pi (via GitHub or USB)
3. Run inference with Hailo + Camera Module 3